# Kiểm thử CNN–LSTM tuần tự trên hai dataset mới

Notebook này **không train lại** mô hình. Nó gộp `obfuscated_grouped.csv` và `xss_payloads_with_obfuscated.csv`, tiền xử lý payload theo cùng pipeline của project, rồi đánh giá bằng **mô hình và tokenizer đang được webapp dùng**. Ô cuối in bảng hiệu quả theo kiểu bảng trong notebook `cnn_only`.

- Nhãn: `0 = Normal`, `1 = Attack`.
- Chỉ bỏ bản ghi gốc trùng **hoàn toàn trong từng dataset** và payload rỗng. Các biến thể obfuscation được giữ lại.
- Payload được bọc bằng `wrap_payload_as_request`, sau đó chuẩn hóa khoảng trắng như lúc train. Không URL decode, HTML unescape hay lowercase.
- Test trên toàn bộ dữ liệu theo mặc định; không dùng dữ liệu mới để fit tokenizer, tune threshold hoặc chọn model.

## 1. Thiết lập

Chạy notebook từ thư mục gốc project hoặc thư mục `cnn_lstm`. Notebook sẽ dùng Python trong `.venv-webapp` nếu có; nếu không, dùng Python của kernel. Môi trường chạy model cần các thư viện trong `webapp/requirements.txt`.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "preprocessing" / "preprocess_data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Hãy chạy notebook từ repo root hoặc thư mục cnn_lstm.")

MODEL_PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv-webapp" / "Scripts" / "python.exe",  # Windows
    PROJECT_ROOT / ".venv-webapp" / "bin" / "python",          # Linux/macOS
]
MODEL_PYTHON = next((path for path in MODEL_PYTHON_CANDIDATES if path.is_file()), Path(sys.executable))
EVALUATOR = PROJECT_ROOT / "analysis" / "evaluate_merged_external.py"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "merged_external_evaluation"
INPUT_FILES = [
    PROJECT_ROOT / "obfuscated_grouped.csv",
    PROJECT_ROOT / "xss_payloads_with_obfuscated.csv",
]
MODEL_FILE = PROJECT_ROOT / "cnn_lstm" / "artifacts_cnn_lstm_tuning" / "obfu_http" / "final" / "best_tuned_hybrid_cnn_lstm.keras"

missing = [path for path in [EVALUATOR, MODEL_FILE, *INPUT_FILES] if not path.is_file()]
if missing:
    raise FileNotFoundError("Thiếu file cần thiết:\n" + "\n".join(map(str, missing)))

BATCH_SIZE = 512
TEST_LIMIT = None  # Đặt số nguyên dương để chạy thử một phần; None = toàn bộ dữ liệu.
print("Project:", PROJECT_ROOT)
print("Python chạy model:", MODEL_PYTHON)
print("Model:", MODEL_FILE.name)
print("Giới hạn test:", TEST_LIMIT or "Toàn bộ dữ liệu")

Project: C:\Users\admin\Desktop\obfuscated-web-attack-detection
Python chạy model: C:\Users\admin\Desktop\obfuscated-web-attack-detection\.venv-webapp\Scripts\python.exe
Model: best_tuned_hybrid_cnn_lstm.keras
Giới hạn test: Toàn bộ dữ liệu


## 2. Gộp, tiền xử lý và chạy test

Ô này gọi `analysis/evaluate_merged_external.py`. Script ghi `merged_preprocessed.csv`, `predictions.csv` và `summary.json` trong `reports/merged_external_evaluation/`. Ngưỡng phân loại lấy từ metadata của chính model webapp (hiện là `0.5`).

In [2]:
command = [
    str(MODEL_PYTHON), str(EVALUATOR),
    "--batch-size", str(BATCH_SIZE),
    "--output-dir", str(OUTPUT_DIR),
]
if TEST_LIMIT is not None:
    if not isinstance(TEST_LIMIT, int) or TEST_LIMIT <= 0:
        raise ValueError("TEST_LIMIT phải là số nguyên dương hoặc None.")
    command += ["--limit", str(TEST_LIMIT)]

with subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Đánh giá thất bại (exit code {return_code}).")

2026-09-17 20:03:46.476575: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-09-17 20:03:47.685880: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-09-17 20:03:50.174226: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Evaluated 10,240/156,186 rows


Evaluated 20,480/156,186 rows


...

Evaluated 133,120/156,186 rows


Evaluated 143,360/156,186 rows


Evaluated 153,600/156,186 rows


Evaluated 156,186/156,186 rows


Saved merged data: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\merged_preprocessed.csv
Saved predictions: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\predictions.csv
Saved metrics: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\summary.json
Overall: {"rows": 156186, "label_counts": {"normal": 74106, "attack": 82080}, "accuracy": 0.8095091749580628, "confusion_matrix_tn_fp_fn_tp": [65208, 8898, 20854, 61226], "normal": {"precision": 0.757686319165253, "recall": 0.8799287507084447, "f1-score": 0.8142450427051596, "support": 74106.0}, "attack": {"precision": 0.8731104899891621, "recall": 0.7459307992202729, "f1-score": 0.8045255052429634, "support": 82080.0}, "macro_avg": {"precision": 0.8153984045772076, "recall": 0.8129297749643588, "f1-score": 0.8093852739740615, "support": 156186.0}, "weighted_avg": {"precision": 0.8183448669302669, "recall": 0.8095091

## 3. Kiểm tra dữ liệu sau tiền xử lý

Kiểm tra số dòng được giữ lại, số dòng trùng hoàn toàn bị bỏ trong từng file và mức trùng chính xác với tập train đã lưu. Các dòng có cùng đầu vào sau chuẩn hóa vẫn được giữ theo yêu cầu.

In [3]:
with (OUTPUT_DIR / "summary.json").open(encoding="utf-8") as file:
    results = json.load(file)

prep = results["preparation"]
source_audit = pd.DataFrame([
    {
        "Dataset": source,
        "Dòng gốc": count,
        "Trùng hoàn toàn đã bỏ": prep["exact_duplicate_rows_removed_by_source"][source],
        "Dòng đã test": prep["evaluated_rows_by_source"][source],
    }
    for source, count in prep["input_rows_by_source"].items()
])
display(source_audit)
print("Payload rỗng đã bỏ:", prep["empty_rows_removed"])
print("Đầu vào giống nhau sau chuẩn hóa vẫn giữ:", prep["repeated_model_inputs_retained"])
print("Trùng chính xác với tập train:", results["training_overlap_rows"] if results["training_overlap_checked"] else "Không có tập train để kiểm tra")
print("Bị cắt ở max_len:", results["truncated_rows"])
print("Đây là kết quả một phần:" , results["partial_test"])

,Dataset,Dòng gốc,Trùng hoàn toàn đã bỏ,Dòng đã test
0,obfuscated_grouped,134778,0,134777
1,xss_payloads_with_obfuscated,21410,0,21409


Payload rỗng đã bỏ: 2
Đầu vào giống nhau sau chuẩn hóa vẫn giữ: 10687
Trùng chính xác với tập train: 0
Bị cắt ở max_len: 5284
Đây là kết quả một phần: False


## 4. Hiệu quả theo kỹ thuật biến đổi

Recall của lớp Attack theo từng kỹ thuật giúp thấy kiểu obfuscation nào mô hình dễ bỏ sót. `unspecified` là dataset XSS không có cột `technique`.

In [4]:
technique_rows = []
for technique, result in results["by_technique"].items():
    matrix = result["confusion_matrix_tn_fp_fn_tp"]
    technique_rows.append({
        "Technique": technique,
        "Rows": result["rows"],
        "Attack Recall": result["attack"]["recall"],
        "Attack F1": result["attack"]["f1-score"],
        "FP": matrix[1],
        "FN": matrix[2],
    })
technique_table = pd.DataFrame(technique_rows).sort_values("Technique").reset_index(drop=True)
display(technique_table.style.format({"Attack Recall": "{:.4%}", "Attack F1": "{:.4%}"}))

,Technique,Rows,Attack Recall,Attack F1,FP,FN
0,case_swapping,7335,65.0579%,78.8304%,0,2563
1,comment_injection,9931,72.1378%,83.8140%,0,2767
2,comment_rewriting,1077,61.7456%,76.3490%,0,412
3,integer_encoding,9462,73.7899%,84.9185%,0,2480
4,logical_invariant,5059,71.0615%,83.0830%,0,1464
5,none,80575,65.6700%,53.0474%,8192,3433
6,operator_swapping,6333,57.8399%,73.2893%,0,2670
7,tautology_change,5059,81.9134%,90.0576%,0,915
8,unspecified,21409,95.7713%,95.9055%,706,756
9,whitespace_substitution,9946,65.8757%,79.4278%,0,3394


## 5. Bảng hiệu quả cuối cùng

Bảng dùng cùng các cột chính với bảng của `cnn_only`: Accuracy, AUC, PR-AUC, threshold, Precision/Recall/F1 của Attack, FP và FN. Dòng `merged_all` là kết quả chính; hai dòng còn lại cho biết hiệu quả theo nguồn.

In [5]:
def performance_row(dataset_name, result):
    tn, fp, fn, tp = result["confusion_matrix_tn_fp_fn_tp"]
    return {
        "Model": "CNN-LSTM tuần tự (WebApp)",
        "Dataset": dataset_name,
        "Rows": result["rows"],
        "Accuracy": result["accuracy"],
        "AUC-ROC": result.get("roc_auc"),
        "PR-AUC": result.get("average_precision"),
        "Threshold": results["threshold"],
        "Attack Precision": result["attack"]["precision"],
        "Attack Recall": result["attack"]["recall"],
        "Attack F1": result["attack"]["f1-score"],
        "FP": fp,
        "FN": fn,
    }

performance_rows = [performance_row("merged_all", results["overall"])]
performance_rows.extend(
    performance_row(name, result) for name, result in results["by_source"].items()
)
performance_table = pd.DataFrame(performance_rows)
performance_table.to_csv(OUTPUT_DIR / "performance_table.csv", index=False, encoding="utf-8")

display(Markdown("### Hiệu quả CNN–LSTM trên dataset đã gộp"))
display(performance_table.style.format({
    "Accuracy": "{:.4%}",
    "AUC-ROC": "{:.4%}",
    "PR-AUC": "{:.4%}",
    "Threshold": "{:.4f}",
    "Attack Precision": "{:.4%}",
    "Attack Recall": "{:.4%}",
    "Attack F1": "{:.4%}",
}))

### Hiệu quả CNN–LSTM trên dataset đã gộp

,Model,Dataset,Rows,Accuracy,AUC-ROC,PR-AUC,Threshold,Attack Precision,Attack Recall,Attack F1,FP,FN
0,CNN-LSTM tuần tự (WebApp),merged_all,156186,80.9509%,90.3319%,89.7560%,0.5000,87.3110%,74.5931%,80.4526%,8898,20854
1,CNN-LSTM tuần tự (WebApp),obfuscated_grouped,134777,79.0098%,88.8527%,84.4628%,0.5000,84.3353%,68.6957%,75.7163%,8192,20098
2,CNN-LSTM tuần tự (WebApp),xss_payloads_with_obfuscated,21409,93.1711%,96.5778%,99.3744%,0.5000,96.0399%,95.7713%,95.9055%,706,756
